# TensorFlow: Classification using transfer learning (without freezing weights, fine-tuning)

In [ ]:
import os
import matplotlib.pyplot as plt

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_docs as tfdocs
import tensorflow_docs.modeling
import tensorflow_docs.plots
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

__Variable definitions__

In [ ]:
# Set image size
ORIGIN_IMAGE_SIZE = (32, 32)
#
SAMPLE_IMAGE_SIZE = (224, 224)
# Set the size of batches
BATCH_SIZE = 32

# Prepare dataset

In [ ]:
(raw_tr_ds, raw_ts_ds), ds_info = tfds.load("cifar10",
    split=["train", "test"],
    with_info=True,
    as_supervised=True)

In [ ]:
def preprocess(image, label):
    image = tf.keras.applications.resnet50.preprocess_input(tf.cast(image, dtype=tf.float32))
    return image, label

In [ ]:
BUFFER_SIZE = 10000

tr_ds = (raw_tr_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(BUFFER_SIZE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

ts_ds = (raw_ts_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE))

# Create model

In [ ]:
inputs = tf.keras.layers.Input(shape=(*ORIGIN_IMAGE_SIZE, 3))
x = tf.keras.layers.UpSampling2D(size=(7,7))(inputs)
x = tf.keras.applications.resnet.ResNet50(
    input_shape=(*SAMPLE_IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet")(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(1024, activation="relu")(x)
x = tf.keras.layers.Dense(512, activation="relu")(x)
outputs = tf.keras.layers.Dense(10, activation="softmax", name="classification")(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
model.compile(
  optimizer="SGD",
  loss="sparse_categorical_crossentropy",
  metrics = ["accuracy"])

# Train model

In [ ]:
history = model.fit(
    tr_ds,
    epochs=4,
    validation_data=ts_ds,
    batch_size=BATCH_SIZE)

# Evaluate model

In [ ]:
loss, accu = model.evaluate(ts_ds, batch_size=BATCH_SIZE)

## Plot history

In [ ]:
plotter = tfdocs.plots.HistoryPlotter(metric="loss", smoothing_std=10)
plotter.plot({"Cifar": history})

## Plot confusion matrix

In [ ]:
it = list(ts_ds.unbatch().as_numpy_iterator())
test_xs = np.array([item[0] for item in it])[:10]
test_ys = np.array([item[1] for item in it])[:10]

In [ ]:
ys_pred = np.argmax(model(test_xs), axis=1)

In [ ]:
labels = ds_info.features["label"].names

# Calculate confusion matrix
cm = confusion_matrix(
    y_true=test_ys,
    y_pred=ys_pred,
    labels=np.arange(len(labels)))

# Create confusion matrix display
ds = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels)

In [ ]:
# Display confusion matrix
fig, ax = plt.subplots(figsize=(8, 8))
ds.plot(colorbar=False, ax=ax)
plt.show()